# model.ipynb
This notebook contains the main code for Deliverable 2 (D2): Machine Learning Model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix

# Load dataset
dataset = pd.read_csv('StudentPerformanceFactors.csv')
dataset.head()


## Separate Variables


In [ ]:
y = dataset.iloc[:, -1]
X = dataset.iloc[:, :-1]
print(X.head())
print(y.head())


## Data Cleaning


In [ ]:
# Handle missing values for numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='mean')
X[numeric_cols] = imputer.fit_transform(X[numeric_cols])

# Remove duplicates
dataset = dataset.drop_duplicates()
dataset.head()


## Encoding Categorical Variables


In [ ]:
for col in dataset.select_dtypes(include=['object']).columns:
    dataset[col] = LabelEncoder().fit_transform(dataset[col])

dataset.head()


## Regression: Feature Selection and Split


In [ ]:
X_reg = dataset[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions']]
y_reg = dataset['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## Feature Scaling


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Train Regression Models


In [ ]:
linear_model = LinearRegression().fit(X_train_scaled, y_train)
poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_model.fit(X_train_scaled, y_train)
ridge_model = Ridge().fit(X_train_scaled, y_train)
lasso_model = Lasso().fit(X_train_scaled, y_train)
elastic_model = ElasticNet().fit(X_train_scaled, y_train)


## Evaluate Regression Models


In [ ]:
models = {
    'Linear Regression': linear_model,
    'Polynomial Regression': poly_model,
    'Ridge': ridge_model,
    'Lasso': lasso_model,
    'Elastic Net': elastic_model,
}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    print(name)
    print('MSE:', mean_squared_error(y_test, y_pred))
    print('R2:', r2_score(y_test, y_pred))
    print('-' * 40)


## Classification Target


In [ ]:
dataset['Performance'] = dataset['Exam_Score'].apply(lambda x: 'Good' if x >= 70 else 'Bad')
dataset[['Exam_Score', 'Performance']].head()


## Classification Models


In [ ]:
X_cls = dataset[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions']]
y_cls = dataset['Performance']

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

cls_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
}

for name, model in cls_models.items():
    model.fit(X_train_cls, y_train_cls)
    y_pred = model.predict(X_test_cls)
    print(name)
    print('Accuracy:', accuracy_score(y_test_cls, y_pred))
    print('Confusion Matrix:\n', confusion_matrix(y_test_cls, y_pred))
    print('-' * 40)


## Clustering (K-Means)


In [ ]:
X_cluster = dataset[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions']]
kmeans = KMeans(n_clusters=3, random_state=42)
dataset['Cluster'] = kmeans.fit_predict(X_cluster)
dataset[['Hours_Studied', 'Attendance', 'Cluster']].head()


## Cluster Visualization


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(dataset['Hours_Studied'], dataset['Attendance'], c=dataset['Cluster'])
plt.xlabel('Hours_Studied')
plt.ylabel('Attendance')
plt.title('K-Means Clustering')
plt.show()
